### Imports

In [ ]:
from qarp.blocks import TrotterBlock, ComputationalBasisStateBlock
from qarp.operators import JordanWigner
from qarp.operators.models import fermi_hubbard
from qarp.algorithms import QPE, find_occupation_numbers, dirichlet_kernel_squared

from qarp.operators.functions import eigenspectrum
from qarp.operators import FullyCommuting
import numpy as np

### QPE object

In [ ]:
# Hamiltonian Trotterization

# QPE parameters
n_qubits = 4
n_ancilla = 5

# Trotterization parameters
time = 2 * np.pi
n_trotter_steps = 4
trotter_order = 2
grouping = FullyCommuting()

fham = fermi_hubbard((2,), t=0.14, U=0.231)
qham = JordanWigner().encode_operator(fham)
eigs = eigenspectrum(qham)
if eigs[0] < 0:
    qham = qham + np.abs(eigs[0]) + np.finfo(float).eps # manual shift to keep the spectrum between 0 and 1

# n=1
# qham = 0.3 + QubitOperator("Z0", 0.2) 
# qham = 0.55 + QubitOperator("Z0", 0.2) 
# n=2
# qham = 0.45 + QubitOperator("Z0 Z1", 0.2) + QubitOperator("X0 Y1", 0.2) 
# n=3
# qham = 0.45 + QubitOperator("Z0 Z1 Z2", 0.2) + QubitOperator("X0 Y1 Y2", 0.1) + QubitOperator("Y0 X1 X2", 0.2) - QubitOperator("X0 Y1", 0.05) - QubitOperator("Y0 X1", 0.05)
# qham = 0.5 + QubitOperator("Z0 Z1 Z2", 0.2) + QubitOperator("X0 Y1 Y2", 0.1) + QubitOperator("Y0 X1 X2", 0.2) - QubitOperator("X0 Y1", 0.05) - QubitOperator("Y0 X1", 0.05)
# qham = QubitOperator("Z0 Z1", 0.2) + QubitOperator("X0 Y1", 0.3) - QubitOperator("Y0 X1", 0.4)

eigs = eigenspectrum(qham)
print("Eigenvalues: ", eigs)

# create the Trotter circuit generated by the Hamiltonian.
# Note the minus sign on the time: TrotterBlock builds U = exp(-iH*time), so we
# pass -time to get U = exp(+iH*time). QPE then reads the phase phi = E directly.
# (With +time the phase comes out mirrored as phi = 1 - E.)
circ_unitary = TrotterBlock(operator=qham, n_qubits=n_qubits, steps=n_trotter_steps, time=-time, order=trotter_order, grouping=grouping).build()

In [ ]:
# show the eigenvalues and relative eigenstate components to prepare a suitable initial state
find_occupation_numbers(qham, n_qubits, verbose=True);

In [ ]:
# build the circuit to prepare |1111>
# this should return a phase as close as possible to 0.6494
circ_state = ComputationalBasisStateBlock([1] * n_qubits)

# create the QPE object
qpe = QPE(circ_state, circ_unitary, n_ancilla).build()

### QPE blocks

In [ ]:
# access the QPE state preparation circuit
qpe.state.plot()

In [ ]:
# access the QPE full circuit
qpe.unitary.plot(scrollable=True)

### Run the algorithm

In [ ]:
# run the QPE sampling
qpe.run()
# access the QPE result
res = qpe.result
# access the QPE result probability
prob = qpe.result_probability

# the phase should be close to 0.6494 because we probed the |1111> state
print(f"The phase is {res:.4f} with probability {prob:.4f}")

### Estimated phase via Dirichlet kernel fit

In [ ]:
# fit_range is the range in which the phase is guessed to be found
# if you are not sure, you can use the whole histogram by setting fit_range to None
# e.g. fit_range = (0.5, 0.7) # if you think the phase is in this range
fit_range = None

phi_estim = qpe.estimate_phase(fit_range=fit_range, verbose=True)
print("The deviation from the true phase is: ", np.abs(phi_estim - eigs[-1]))

### Plot

In [ ]:
# plot the distribution
fig, ax = qpe.plot(return_fig=True)

ax.plot(qpe.freqs, dirichlet_kernel_squared(qpe.freqs, phi_estim, 2**n_ancilla), 'r', label="Dirichlet kernel squared")